In [0]:
# Score the latest feature row of every symbol with the champion model and append to the inference log.
# The log carries everything the Lakehouse inference monitor needs: features, prediction,
# actual_next_close (back-filled later by backfill_labels.py), request_ts and model_version.
import sys
import os
sys.path.insert(0, os.getcwd())

import mlflow
from mlflow import MlflowClient
from pyspark.sql import functions as F
from utils import FEATURE_COLS, load_project, load_yaml, model_name as qualified_model_name, table, widget_or

mlflow.set_registry_uri("databricks-uc")
cfg = load_project()

dbutils.widgets.text("catalog", "stock_demo")
dbutils.widgets.text("model_alias", "champion")
dbutils.widgets.text("run_source", "batch")
catalog = dbutils.widgets.get("catalog")
alias = dbutils.widgets.get("model_alias")
run_source = widget_or(dbutils.widgets.get("run_source"), "batch")

name = qualified_model_name(cfg, catalog)
try:
    version = str(MlflowClient().get_model_version_by_alias(name, alias).version)
except Exception as e:
    raise RuntimeError(
        f"No '{alias}' model found for {name}. Run stock_training_job first and check that evaluate_model passed "
        f"(a failed quality gate skips registration). Original error: {e}") from e
model = mlflow.pyfunc.load_model(f"models:/{name}@{alias}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.serving")
features = spark.table(table(cfg, "features", catalog))
latest = features.groupBy("symbol").agg(F.max("trade_ts").alias("max_ts"))
scoring = features.join(latest, "symbol").filter(F.col("trade_ts") == F.col("max_ts")).drop("max_ts")

# 1) Only score symbols with FRESH data: with live data for a subset of symbols, the others stop in the past
#    and a "prediction for tomorrow" for them would be meaningless.
max_staleness_days = int(load_yaml("live.yml")["live_data"]["max_staleness_days"])
newest = features.agg(F.max("trade_ts")).collect()[0][0]
scoring = scoring.filter(F.col("trade_ts") >= F.lit(newest) - F.expr(f"INTERVAL {max_staleness_days} DAYS"))

# 2) Idempotent: skip (symbol, day) pairs already scored by this model version in this run_source.
log_table = table(cfg, "inference_log", catalog)
if spark.catalog.tableExists(log_table):
    done = (spark.table(log_table)
            .filter((F.col("model_version") == version) & (F.col("run_source") == run_source))
            .select("symbol", "trade_ts").distinct())
    scoring = scoring.join(done, ["symbol", "trade_ts"], "left_anti")

if scoring.limit(1).count() == 0:
    print(f"Nothing new to score: every fresh symbol-day is already in {log_table} for model v{version}.")
    dbutils.notebook.exit("no new rows")

pdf = scoring.select("symbol", "trade_ts", *FEATURE_COLS).toPandas()
pdf[FEATURE_COLS] = pdf[FEATURE_COLS].astype("float64")
pdf["prediction"] = model.predict(pdf[FEATURE_COLS])

out = spark.createDataFrame(pdf)
for c in FEATURE_COLS:
    out = out.withColumn(c, F.col(c).cast("double"))
out = (out
       .withColumn("actual_next_close", F.lit(None).cast("double"))   # back-filled once the next session exists
       .withColumn("request_ts", F.current_timestamp())
       .withColumn("model_version", F.lit(version))
       .withColumn("model_alias", F.lit(alias))
       .withColumn("run_source", F.lit(run_source)))

out.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(log_table)
# Required by Lakehouse Monitoring when the monitor computes drift/changes over time
spark.sql(f"ALTER TABLE {log_table} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

print(f"Logged {out.count()} predictions from {name}@{alias} (v{version}) to {log_table}")
display(spark.table(log_table).orderBy(F.desc("request_ts")).limit(50))